# ReLoop Grading — Colab GPU training

Real fine-tune run for the DINOv2 condition grader on a **free T4 GPU**.
Unlike the local M1 path (frozen backbone), this **unfreezes the last 4 DINOv2 blocks**,
turns on the SOP viewpoint-invariance loss, pulls more data, and can add real
MVTec/VisA defects.

**How to run:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.
In `github` mode (default) it clones the repo automatically — no uploads. Switch the
top cell to `upload` if you'd rather drag in the zip.

Honest note: without a hand-labelled gold set, grade/F1 are measured on a synthetic
val split — treat the cosine-similarity separation + loss convergence as the real signal.

In [ ]:
# === Settings ===
MODE = "github"            # "github" (clone the repo) or "upload" (drag the zip)
REPO_URL = "https://github.com/paawankhurana2005/hack"
BRANCH   = "feat/ml-extensiveness-p0-p6"
ADD_REAL_DEFECTS = False  # True -> also download VisA (Amazon) for real defect supervision
SAVE_TO_DRIVE    = False  # True -> copy the checkpoint to your Google Drive at the end

In [ ]:
# === Get the code ===
import os, glob, zipfile
if MODE == "upload":
    from google.colab import files
    up = files.upload()                      # drag reloop_grading_colab.zip here
    zname = [k for k in up if k.endswith('.zip')][0]
    with zipfile.ZipFile(zname) as z: z.extractall('reloop_pkg')
    GRADING_DIR = os.path.dirname(glob.glob('reloop_pkg/**/requirements.txt', recursive=True)[0])
else:
    os.system(f"git clone -b {BRANCH} {REPO_URL} reloop_repo")
    GRADING_DIR = 'reloop_repo/ml/grading'
os.chdir(GRADING_DIR)
print('working dir:', os.getcwd())
print('contents:', os.listdir('.'))

In [ ]:
# === Install deps (Colab already has torch/torchvision) ===
!pip -q install 'transformers>=4.40' 'scikit-learn>=1.3' pyyaml pillow tqdm

In [ ]:
# === GPU check ===
import torch
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (set Runtime->T4 GPU!)')

In [ ]:
# === (Optional) real defect data: VisA (Amazon Science) ===
DATA_ROOT = ''
if ADD_REAL_DEFECTS:
    os.makedirs('data', exist_ok=True)
    rc = os.system('wget -q https://amazon-visual-anomaly.s3.us-west-2.amazonaws.com/VisA_20220922.tar -O data/visa.tar && tar -xf data/visa.tar -C data')
    if rc == 0:
        DATA_ROOT = './data'; print('VisA ready under ./data')
    else:
        print('VisA download failed (URL may have changed) — continuing without it.')
    print('MVTec AD: free form download at https://www.mvtec.com/company/research/datasets/mvtec-ad — unpack into ./data to include it.')
else:
    print('Training on ABO + synthetic + SOP (defect supervision = synthetic). MVTec/VisA adapters skip gracefully.')

In [ ]:
# === TRAIN (unfreezes the DINOv2 tail; streams live logs) ===
# get_ipython().system streams the child's stdout into THIS cell (os.system did not),
# and `-u` makes Python unbuffered so you see [train]/[download] lines in real time.
extra = f' --data-root {DATA_ROOT}' if DATA_ROOT else ''
get_ipython().system(f'python -u -m reloop_grading.train --config configs/colab.yaml{extra}')

In [ ]:
# === EVALUATE (accuracy, macro-F1, defect-F1, confusion, ECE, similarity dist) ===
import json
get_ipython().system('python -u -m reloop_grading.evaluate --checkpoint runs/colab/grading_model.pt --device cuda --out eval_report.json')
print(json.dumps(json.load(open('eval_report.json')), indent=2))

In [ ]:
# === STRUCTURED INFERENCE demo (single image + reference diff) ===
import glob, json
from reloop_grading.inference import GradingInference
inf = GradingInference.from_checkpoint('runs/colab/grading_model.pt', 'cuda')
imgs = sorted(glob.glob('.cache/reloop_grading/abo/*.jpg'))
print('single-image :', json.dumps(inf.grade_json(imgs[0]), indent=2))
if len(imgs) >= 2:
    print('with-reference:', json.dumps(inf.grade_json(imgs[1], original=imgs[0]), indent=2))

In [ ]:
# === (Optional) persist the checkpoint to Google Drive ===
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    os.makedirs('/content/drive/MyDrive/reloop_grading', exist_ok=True)
    os.system('cp runs/colab/grading_model.pt eval_report.json /content/drive/MyDrive/reloop_grading/')
    print('saved to Drive/MyDrive/reloop_grading/')
else:
    print('SAVE_TO_DRIVE=False — download runs/colab/grading_model.pt from the Files panel if you want to keep it.')